# Live hard coral: model predictions vs CoralNet annotations, image by image (10 sources)

Same as `Coral_Cover_Qualitative_10src.ipynb` (same 10 CoralNet **test** sources, same model `cbm_scratch100k_base_taxv2`
at step 100k, 256 px, **same sampled images**), with hard coral restricted to **live** hard coral on both sides:
- **Model**: P(live hard coral) = P(order = scleractinia) × P(live), the order head's softmax times the sigmoid `live` channel.
  A point or pixel counts as predicted live hard coral when this product is ≥ 0.5 (the other notebook uses the order argmax).
- **Annotations**: live hard coral = the CoralNet class maps to `order = scleractinia` **and** `live` positive in the run's
  `class_to_concepts.csv`; negative when either is negative (dead coral is not live hard coral); otherwise not given.
  In these 10 sources every annotated hard coral point is also live (the sources were chosen to exclude those where dead coral
  maps to scleractinia), so the annotations match the other notebook and only the model side changes.

**What each figure shows**: a 2 × 2 grid with the bare image (top left), **P(order = scleractinia)** (top right),
**P(live)** (bottom left) and their product **P(scleractinia) × P(live)** (bottom right), so you can see which factor
drives the product. Each panel title gives that panel's cover at the points and over all pixels.
- **Shading**: each panel's probability in magenta bands: 0.25–0.5 faint, 0.5–0.75 medium, 0.75–1 strong
  (nothing below 0.25); lines at P = 0.25 (dotted), 0.5 (dashed), 0.75 (solid).
- **Markers**: fill = annotation (yellow live hard coral, blue not, grey not given / unmapped); magenta ring = that
  panel's call at the point (order argmax for scleractinia, P ≥ 0.5 for live and for the product).
- **Labels** (product panel): CoralNet class, with its MERMAID class in brackets.
- **Title**: live hard coral cover from the annotations, from the model at the same points, and from the model over all pixels.

Batch version:
```
python scripts/qualitative_overlays.py --target live_hard --panels hard live live_hard --show-raw \
    --n-per-source 100 \
    --out eval_out/coral_cover/qualitative/base_taxv2_live
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display

REPO = Path("..").resolve() if Path("../scripts").exists() else Path(".").resolve()
load_dotenv(REPO / ".env")
sys.path.insert(0, str(REPO / "scripts"))
import qualitative_overlays as qo  # noqa: E402

RUN = "cbm_scratch100k_base_taxv2"
SOURCES = qo.SOURCES_10
N_PER_SOURCE = 1   # a few images to settle the figure; the CLI does 100 per source
SEED = 0
TARGET = "live_hard"  # P(scleractinia) x P(live); "hard" reproduces Coral_Cover_Qualitative_10src
MERMAID_MAP = "csv"  # "json": configs/coralnet_to_mermaid_mapping_temporary.json instead
BANDS = ((0.25, 0.5, 0.1), (0.5, 0.75, 0.3), (0.75, 1.0, 0.45))  # (low, high, opacity) of P(hard coral)
CONTOURS = ((0.25, ":"), (0.5, "--"), (0.75, "-"))  # (level, linestyle) iso-lines
PANELS = ("hard", "live", "live_hard")  # one panel per map; ("live_hard",) for the product only
STYLE = dict(bands=BANDS, contours=CONTOURS, panels=PANELS, show_raw=True, label_points=True, fontsize=6.0, marker_size=60, width_in=18)

Load the model and the 10 sources once (the CoralNet manifest read takes a minute or two).

In [ ]:
ctx = qo.load_context(RUN, SOURCES, mermaid_map=MERMAID_MAP, target=TARGET)
sample = qo.sample_images(ctx, N_PER_SOURCE, seed=SEED)
sample

## One image per source

In [ ]:
preds = []
for _, meta in sample.iterrows():
    pred = qo.predict_image(ctx, meta)
    preds.append(pred)
    fig = qo.render(pred, **STYLE)
    plt.show()

Per-point table: CoralNet class, MERMAID class, annotation code for live hard coral (2 yes, 1 no, 0 not given) with its two parts (`hard_code`, `live_code`), and the model's P(scleractinia), P(live) and their product `p_hard`.

In [ ]:
import pandas as pd

points = pd.concat([p["points"].assign(source_id=p["summary"]["source_id"], image_id=p["summary"]["image_id"])
                    for p in preds], ignore_index=True)
summary = pd.DataFrame([p["summary"] for p in preds])
display(summary)
display(points[["source_id", "image_id", "coralnet_name", "mermaid", "gt_code", "hard_code", "live_code",
                "p_scleractinia", "p_live", "p_live_hard", "pred_order", "call_hard", "call_live", "call_live_hard"]])

## Disagreements only

Points where the annotation and the model (product ≥ 0.5) disagree on live hard coral, and the CoralNet → MERMAID label pairs
behind them. Useful for spotting mapping problems separately from model errors.

In [ ]:
gt_live = points.gt_code == 2
dis = points[(gt_live != points.pred_hard) & (points.gt_code != 0)]
(dis.assign(kind=lambda d: d.pred_hard.map({True: "model live hard, GT not", False: "GT live hard, model not"}))
    .groupby(["kind", "coralnet_name", "mermaid"]).size().rename("points").sort_values(ascending=False).to_frame())

## Where `live` changes the call

Points whose order argmax is scleractinia (hard coral in the other notebook) but whose P(scleractinia) × P(live) is below 0.5,
i.e. the model sees coral but not live coral. Each row is one such point, with its annotation.

In [ ]:
flip = points[(points.pred_order == "scleractinia") & ~points.pred_hard]
print(f"{len(flip)} of {(points.pred_order == 'scleractinia').sum()} points predicted scleractinia drop out with live")
flip[["source_id", "image_id", "coralnet_name", "mermaid", "gt_code", "p_scleractinia", "p_live", "p_live_hard"]]

## Inspect a single image

Re-render any image, e.g. without labels or with different bands.

In [ ]:
qo.render(preds[0], **{**STYLE, "label_points": False})
plt.show()